## Run_papermill

Automates the execution of notebooks — useful for running the same notebook across a series of datasets without manually re-running it each time.

#### How does it work?

You need a **parent notebook** (the template) and a **series of datasets** to process.

1. In the parent notebook, tag one cell as `parameters`. This cell should define all the parameters you want papermill to override (e.g. the dataset path, and any other setting that changes from one dataset to the next).
2. For each dataset, call `papermill.execute_notebook()`, passing in the values for these parameters. Papermill injects them into the tagged cell and runs a new copy of the notebook — your parent notebook itself is left untouched, and a separate output notebook is generated per dataset (not saved if output_path==None).

This is typically done in a loop (e.g. one `execute_notebook` call per dataset path), so you end up with one executed notebook per dataset instead of manually changing parameters and re-running the same notebook by hand.

#### Recommendation for the s3dxrd / friedel_pairs pipeline

Some steps of the pipeline (via `pf3dxrd` and `ImageD11.friedel_pairs`) are designed to be submitted as **SLURM batch jobs** rather than run interactively — this includes Friedel pairing (`fp1`) and indexing (`fp3`). For these steps, submitting to SLURM directly (rather than using papermill) is recommended, because:

- You can allocate more CPUs and more memory per task than a local/interactive Jupyter session would allow.
- Once the jobs are submitted, you don't need to keep a Jupyter session (or your own machine) active while they run — SLURM handles execution and queuing independently.

Other parts of the pipeline are not yet set up to submit to SLURM this way. For those steps, **papermill** is the more practical option: it lets you batch-run a notebook across datasets without submitting a formal SLURM job, but you need to maintain an active session during execution.

In [2]:
import os, sys, time, glob
start = time.time()

# python environment stuff
IMAGED11_PATH = '/home/esrf/jean1994b/ImageD11_jbjacob'  # None means do not use git, otherwise enter the name of the folder to use for the git checkout "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = 'ImageD11'  # the name of the git checkout folder within path. None means guess


if IMAGED11_PATH is not None:
    if '/data/id11/nanoscope' not in sys.path:
        sys.path.append('/data/id11/nanoscope')
    import install_ImageD11_from_git
    install_ImageD11_from_git.setup_ImageD11_from_git(IMAGED11_PATH,CHECKOUT_PATH)

# Setting path via: 
sys.path.insert(0, /home/esrf/jean1994b/ImageD11_jbjacob/ImageD11 )
# Running from: /home/esrf/jean1994b/ImageD11_jbjacob/ImageD11/ImageD11/__init__.py


In [3]:
import ImageD11.sinograms.dataset
import papermill

%load_ext autoreload
%autoreload 2

In [5]:
# parent notebook
notebook = 'fp4_grain_mapping.ipynb'

# project root
root = '/data/visitor/es1832/id11/20260715/PROCESSED_DATA/'

In [8]:
# sample list
skip=['json', 'ipynb']
samples = [p for p in os.listdir(root) if ('MgO' in p) and all([sk not in p for sk in skip])]
samples

['MgO_14_0p3M',
 'MgO_11_DI',
 'MgO_14_0p3M_b',
 'MgO_12_0p2M',
 'MgO_3_0p1M',
 'MgO_TRANSFER']

In [9]:
# dataset dict
skip = ['ipynb', 'match','pct', 'pycache','json', 'svg', 'cif']  # skip these

process_dict = {}

for s in samples:
    sample_dir = os.path.join(root,s)
    dsets = sorted([ds for ds in os.listdir(sample_dir) if all([sk not in ds for sk in skip])])
    process_dict[s] = dsets

process_dict

{'MgO_14_0p3M': ['MgO_14_0p3M_12um'],
 'MgO_11_DI': ['MgO_11_DI_12um'],
 'MgO_14_0p3M_b': ['MgO_14_0p3M_b_12um'],
 'MgO_12_0p2M': ['MgO_12_0p2M_12um'],
 'MgO_3_0p1M': ['MgO_3_0p1M_12um_0003', 'combined'],
 'MgO_TRANSFER': ['MgO_14_0p3M_b', 'MgO_3_0p1M']}

In [6]:
# helper function to get dataset fiule path from dataset name 
def get_dsfile(dataroot, sample, dsname):
    return os.path.join(root, sample, dsname, dsname+'_dataset.h5')

In [11]:
# run notebook for all datasets in process dict
for spl in samples:
    for dset in process_dict[spl]:
        dsfile = get_dsfile(root,spl,dset)
        papermill.execute_notebook(notebook,
                                   output_path=None,
                                   parameters={'dsfile':dsfile, 'phase':'MgO'}
                                  )

Executing:   0%|          | 0/38 [00:00<?, ?cell/s]

Executing:   0%|          | 0/38 [00:00<?, ?cell/s]

Executing:   0%|          | 0/38 [00:00<?, ?cell/s]

Executing:   0%|          | 0/38 [00:00<?, ?cell/s]